## 🎯 Learning Objectives
* Understand the critical need for cost optimization in production-grade Generative AI applications, especially RAG systems.
* Learn how caching mechanisms can reduce redundant LLM calls, improve latency, and lower operational costs.
* Explore strategies for model routing to intelligently select the most cost-effective and performant LLM for a given task.
* Grasp the concept of batching requests to LLMs and its impact on throughput and cost efficiency.
* Implement basic examples of caching, model routing, and batching in Python to optimize simulated LLM interactions.


## Cost Optimization in Generative AI: Caching, Model Routing, and Batching

As Generative AI applications, particularly those leveraging Retrieval Augmented Generation (RAG), move from proof-of-concept to production, managing operational costs becomes paramount. Each interaction with a Large Language Model (LLM) incurs a cost, whether it's an API call to a cloud provider (e.g., OpenAI, Anthropic, Google Gemini) or compute cycles on self-hosted models. Without careful optimization, these costs can quickly escalate, impacting profitability and scalability.

This lesson explores three fundamental strategies to significantly reduce the operational expenses and improve the performance of your GenAI applications: **caching**, **model routing**, and **batching**.

### 1. Caching: The Memory Saver

Imagine you're running a popular restaurant. If every customer ordered the same dish, and your chefs had to prepare it from scratch every single time, it would be slow and wasteful. Instead, you might pre-prepare popular components or even entire dishes, storing them for quick retrieval. This is the essence of caching.

In GenAI, caching involves storing the results of expensive LLM calls (or intermediate computations) so that subsequent, identical requests can be served instantly from memory or storage, rather than re-executing the LLM. This drastically reduces API costs, improves response times, and lessens the load on your LLM providers or infrastructure.

**How it helps RAG:**
*   **Query Rewriting/Expansion:** If a user asks the same or a very similar question, the rewritten query can be cached.
*   **Retrieval Results:** If the same query leads to the same document chunks, these can be cached.
*   **Generation:** If a common question or a specific prompt always yields the same LLM response, that response can be cached.

### 2. Model Routing: The Smart Dispatcher

Continuing with our restaurant analogy, imagine you have a team of chefs. One chef is incredibly fast and affordable but specializes in simple dishes. Another is a Michelin-star chef, expensive and slower, but capable of creating complex culinary masterpieces. You wouldn't ask the Michelin-star chef to make a simple salad, would you? You'd route the order to the appropriate chef.

Model routing applies this principle to LLMs. It involves dynamically selecting the most appropriate LLM for a given task based on factors like complexity, cost, latency requirements, and specific capabilities. For instance, a simple summarization task might go to a smaller, cheaper, and faster model, while a complex reasoning or creative writing task is routed to a more powerful, albeit more expensive, model.

**How it helps RAG:**
*   **Simple vs. Complex Queries:** Route simple factual questions to a cheaper model, and nuanced, multi-turn conversations to a more advanced one.
*   **Tool Use/Function Calling:** Use a specialized, smaller model for deciding which tools to call, and a larger model for the final synthesis.
*   **Cost vs. Quality Trade-off:** Prioritize cost for internal, less critical tasks, and quality for customer-facing interactions.

### 3. Batching: The Bulk Processor

Think about shipping packages. Sending 100 individual small packages one by one is far less efficient and more expensive than consolidating them into a single large shipment. Each individual shipment incurs overhead (labeling, pickup, processing).

Batching in GenAI means grouping multiple independent requests into a single API call or inference request to the LLM. Many LLM APIs and inference engines are optimized to process multiple prompts concurrently within a single request, amortizing the fixed overhead associated with each call (e.g., network latency, model loading, initial processing). This significantly improves throughput and reduces the per-item cost.

**How it helps RAG:**
*   **Document Processing:** When processing multiple retrieved documents for summarization or extraction, batch them into a single LLM call.
*   **Parallel Queries:** If a RAG system needs to answer several related questions, or process multiple user inputs simultaneously, batching can be highly effective.
*   **Embedding Generation:** When generating embeddings for a large corpus of documents, batching input texts to the embedding model is a standard practice.

By strategically implementing these three techniques, developers can build more efficient, scalable, and cost-effective Generative AI applications, ensuring that the power of LLMs is harnessed responsibly and economically.


In [ ]:
import time
import functools
import hashlib
import json

# --- Configuration for simulated costs and latencies ---
# These values are illustrative and would be actual API costs/latencies in a real system
COST_PER_TOKEN_GPT35 = 0.0000015 # Example: $1.50 per 1M tokens
COST_PER_TOKEN_GPT4 = 0.00003   # Example: $30 per 1M tokens

LATENCY_PER_TOKEN_GPT35 = 0.005 # 5ms per token
LATENCY_PER_TOKEN_GPT4 = 0.01   # 10ms per token

# --- Mock LLM Functions ---
# These simulate actual LLM API calls

def _calculate_cost_and_latency(prompt_tokens, completion_tokens, model_type):
    """Helper to calculate simulated cost and latency."""
    if model_type == "gpt-3.5-turbo":
        cost = (prompt_tokens * COST_PER_TOKEN_GPT35) + (completion_tokens * COST_PER_TOKEN_GPT35)
        latency = (prompt_tokens * LATENCY_PER_TOKEN_GPT35) + (completion_tokens * LATENCY_PER_TOKEN_GPT35)
    elif model_type == "gpt-4-turbo":
        cost = (prompt_tokens * COST_PER_TOKEN_GPT4) + (completion_tokens * COST_PER_TOKEN_GPT4)
        latency = (prompt_tokens * LATENCY_PER_TOKEN_GPT4) + (completion_tokens * LATENCY_PER_TOKEN_GPT4)
    else:
        raise ValueError("Unknown model type")
    return cost, latency

def mock_llm_gpt35(prompt: str) -> str:
    """Simulates a call to a faster, cheaper LLM (e.g., GPT-3.5 Turbo)."""
    prompt_tokens = len(prompt.split())
    completion_tokens = max(20, len(prompt.split()) // 2) # Simulate some output
    cost, latency = _calculate_cost_and_latency(prompt_tokens, completion_tokens, "gpt-3.5-turbo")

    time.sleep(latency) # Simulate network and processing time
    print(f"[GPT-3.5] Processed '{prompt[:30]}...' | Cost: ${cost:.6f} | Latency: {latency:.4f}s")
    return f"Response from GPT-3.5 for: {prompt}. (Cost: ${cost:.6f})"

def mock_llm_gpt4(prompt: str) -> str:
    """Simulates a call to a slower, more expensive LLM (e.g., GPT-4 Turbo)."""
    prompt_tokens = len(prompt.split())
    completion_tokens = max(50, len(prompt.split())) # Simulate more detailed output
    cost, latency = _calculate_cost_and_latency(prompt_tokens, completion_tokens, "gpt-4-turbo")

    time.sleep(latency) # Simulate network and processing time
    print(f"[GPT-4] Processed '{prompt[:30]}...' | Cost: ${cost:.6f} | Latency: {latency:.4f}s")
    return f"Response from GPT-4 for: {prompt}. (Cost: ${cost:.6f})"

def mock_llm_batched(prompts: list[str], model_type: str) -> list[str]:
    """Simulates a batched call to an LLM, amortizing overhead."""
    if not prompts: return []

    total_prompt_tokens = sum(len(p.split()) for p in prompts)
    total_completion_tokens = sum(max(20, len(p.split()) // 2) for p in prompts) # Simplified

    # Simulate a fixed overhead + per-token cost/latency
    fixed_overhead_latency = 0.1 # 100ms fixed overhead for batch
    fixed_overhead_cost = 0.00001 # $0.00001 fixed overhead for batch

    cost_per_token, latency_per_token = 0, 0
    if model_type == "gpt-3.5-turbo":
        cost_per_token = COST_PER_TOKEN_GPT35
        latency_per_token = LATENCY_PER_TOKEN_GPT35
    elif model_type == "gpt-4-turbo":
        cost_per_token = COST_PER_TOKEN_GPT4
        latency_per_token = LATENCY_PER_TOKEN_GPT4
    else:
        raise ValueError("Unknown model type")

    total_cost = fixed_overhead_cost + (total_prompt_tokens * cost_per_token) + (total_completion_tokens * cost_per_token)
    total_latency = fixed_overhead_latency + (total_prompt_tokens * latency_per_token) + (total_completion_tokens * latency_per_token)

    time.sleep(total_latency)
    print(f"[BATCHED {model_type.upper()}] Processed {len(prompts)} prompts | Total Cost: ${total_cost:.6f} | Total Latency: {total_latency:.4f}s")
    return [f"Batched response for: {p}. (Batch Cost: ${total_cost/len(prompts):.6f} per item)" for p in prompts]

# --- 1. Caching Implementation ---

# A simple in-memory cache using functools.lru_cache
@functools.lru_cache(maxsize=128)
def cached_llm_call(prompt: str, model_func) -> str:
    """A cached LLM call wrapper."""
    # The model_func itself is not part of the cache key by default with lru_cache
    # To make it part of the key, we'd need to pass its name or a unique identifier
    # For this demo, we'll assume model_func is consistent for a given cache.
    print(f"[CACHE MISS] Calling LLM for: {prompt[:30]}...")
    return model_func(prompt)

print("\n--- Demonstrating Caching ---")
start_time = time.perf_counter()
response1 = cached_llm_call("What is the capital of France?", mock_llm_gpt35)
end_time = time.perf_counter()
print(f"Time taken (first call): {end_time - start_time:.4f}s\n")

start_time = time.perf_counter()
response2 = cached_llm_call("What is the capital of France?", mock_llm_gpt35) # Cache hit
end_time = time.perf_counter()
print(f"Time taken (second call, cache hit): {end_time - start_time:.4f}s\n")

start_time = time.perf_counter()
response3 = cached_llm_call("Tell me a short story about a robot.", mock_llm_gpt35) # Cache miss
end_time = time.perf_counter()
print(f"Time taken (third call, new prompt): {end_time - start_time:.4f}s\n")

# --- 2. Model Routing Implementation ---

def route_llm_call(prompt: str) -> str:
    """Routes the LLM call based on prompt complexity (simplified)."""
    if len(prompt.split()) < 15 and "complex" not in prompt.lower():
        print("[ROUTER] Routing to GPT-3.5 (simple query)")
        return mock_llm_gpt35(prompt)
    else:
        print("[ROUTER] Routing to GPT-4 (complex query)")
        return mock_llm_gpt4(prompt)

print("\n--- Demonstrating Model Routing ---")

# Simple query
start_time = time.perf_counter()
response_simple = route_llm_call("What is the highest mountain in the world?")
end_time = time.perf_counter()
print(f"Time taken (simple query): {end_time - start_time:.4f}s\n")

# Complex query
start_time = time.perf_counter()
response_complex = route_llm_call("Explain the concept of quantum entanglement in simple terms, focusing on its implications for future computing, and provide a complex analogy.")
end_time = time.perf_counter()
print(f"Time taken (complex query): {end_time - start_time:.4f}s\n")

# --- 3. Batching Implementation ---

print("\n--- Demonstrating Batching ---")

prompts_to_batch = [
    "Summarize the key points of the latest AI research.",
    "What are the main benefits of cloud computing?",
    "Explain the concept of blockchain in one paragraph.",
    "Give me three ideas for a healthy breakfast."
]

# Without batching (individual calls)
print("\n--- Without Batching ---")
individual_start_time = time.perf_counter()
individual_responses = [mock_llm_gpt35(p) for p in prompts_to_batch]
individual_end_time = time.perf_counter()
individual_total_time = individual_end_time - individual_start_time
print(f"Total time without batching: {individual_total_time:.4f}s\n")

# With batching
print("\n--- With Batching ---")
batched_start_time = time.perf_counter()
batched_responses = mock_llm_batched(prompts_to_batch, "gpt-3.5-turbo")
batched_end_time = time.perf_counter()
batched_total_time = batched_end_time - batched_start_time
print(f"Total time with batching: {batched_total_time:.4f}s\n")

print(f"\nBatching saved approximately {individual_total_time - batched_total_time:.4f}s")

# --- Combined Example (Conceptual) ---
# In a real RAG system, these would be integrated into a pipeline.
# For instance, a RAG query might first hit a cache.
# If not found, it might be routed to a model for query expansion.
# The expanded queries might then be batched for embedding generation.
# Finally, the retrieved context and original query might be batched for final answer generation.

print("\n--- Combined Conceptual Flow ---")
def rag_optimized_flow(user_query: str):
    print(f"Processing query: '{user_query}'")

    # 1. Caching for initial query response
    cached_response = cached_llm_call(user_query, mock_llm_gpt35) # Try GPT-3.5 first
    if "[CACHE MISS]" not in cached_response:
        print("-> Served from cache!")
        return cached_response

    # 2. Model Routing for query expansion/generation
    print("-> Cache miss. Routing for deeper processing...")
    if len(user_query.split()) < 10:
        print("-> Simple query, using GPT-3.5 for initial generation.")
        llm_response = mock_llm_gpt35(user_query)
    else:
        print("-> Complex query, using GPT-4 for detailed generation.")
        llm_response = mock_llm_gpt4(user_query)

    # In a real RAG, this would involve retrieval, then potentially batching for context summarization
    # For demo, we'll just return the LLM response.
    return llm_response

print("\nFirst query (cache miss, simple routing):")
rag_optimized_flow("What is the capital of Japan?")

print("\nSecond query (cache hit):")
rag_optimized_flow("What is the capital of Japan?")

print("\nThird query (cache miss, complex routing):")
rag_optimized_flow("Explain the economic impact of AI on global labor markets over the next decade.")


### Interpreting the Code Output and Trade-offs

The code demonstrates the core principles of caching, model routing, and batching using simulated LLM calls. Let's break down the output and discuss the practical implications.

#### Caching

*   **Output Interpretation:** You'll observe that the first call to `cached_llm_call` for "What is the capital of France?" shows `[CACHE MISS]` and a noticeable `Time taken`. The second identical call, however, executes almost instantaneously, without printing `[CACHE MISS]`, indicating a cache hit. A new, different prompt will again result in a cache miss and a longer execution time.
*   **Performance Trade-offs:**
    *   **Pros:** Dramatically reduces latency and API costs for repeated queries. Reduces load on LLM providers.
    *   **Cons:**
        *   **Cache Invalidation:** Deciding when cached data becomes stale is crucial. If the underlying data or model capabilities change, cached responses might become inaccurate. Strategies include time-based expiration (TTL), event-driven invalidation, or explicit invalidation.
        *   **Memory Usage:** Caches consume memory. `functools.lru_cache` is in-memory, suitable for single-process applications. For distributed systems, external caches like Redis or Memcached are necessary.
        *   **Cache Key Design:** The effectiveness depends on how well the cache key (e.g., the prompt string) captures the uniqueness of a request. Minor prompt variations might lead to cache misses even if the semantic intent is the same.
*   **Typical Use Cases in RAG:** Frequently asked questions, common summarization tasks, query rewriting results, embedding lookups for popular documents.

#### Model Routing

*   **Output Interpretation:** The `[ROUTER]` messages clearly show which mock LLM (`GPT-3.5` or `GPT-4`) was selected based on the prompt's complexity (here, simply word count and a keyword check). You'll see the `Time taken` for the simple query is much lower, reflecting the faster `mock_llm_gpt35`, while the complex query takes longer due to `mock_llm_gpt4`.
*   **Performance Trade-offs:**
    *   **Pros:** Significant cost savings by using cheaper models for simpler tasks. Improved overall latency by leveraging faster models where appropriate. Access to specialized models for specific tasks.
    *   **Cons:**
        *   **Routing Logic Complexity:** Designing effective routing rules can be challenging. Simple heuristics (like prompt length) might be insufficient. More advanced methods involve using a small, fast LLM to classify the query, or a dedicated machine learning model.
        *   **Maintenance Overhead:** Integrating and managing multiple LLM APIs or models adds complexity to your codebase and deployment.
        *   **Suboptimal Routing:** Incorrect routing can lead to poor quality responses (if a simple model handles a complex task) or unnecessary costs (if an expensive model handles a simple task).
*   **Typical Use Cases in RAG:** Classifying user intent (e.g., factual question vs. creative request), determining if a query requires tool use, routing to different models for different stages of the RAG pipeline (e.g., query expansion vs. final answer generation).

#### Batching

*   **Output Interpretation:** The `--- Without Batching ---` section shows individual calls to `mock_llm_gpt35`, each incurring its own simulated latency. The `--- With Batching ---` section shows a single, longer call to `mock_llm_batched`, but the *total time* for processing all prompts is significantly reduced. The final print statement quantifies the time saved.
*   **Performance Trade-offs:**
    *   **Pros:** Amortizes fixed API/inference overheads across multiple requests, leading to higher throughput and lower per-item cost. Ideal for scenarios with many concurrent or queued requests.
    *   **Cons:**
        *   **Increased Latency for Individual Items:** While total throughput improves, the latency for any *single* item within a batch might increase because it has to wait for the entire batch to be ready and processed. This is a critical consideration for real-time, low-latency applications.
        *   **Batch Management Complexity:** Requires logic to collect requests into batches, manage batch sizes, and handle partial failures or timeouts.
        *   **Resource Utilization:** If batches are not consistently full, you might be paying for idle capacity or waiting unnecessarily.
*   **Typical Use Cases in RAG:** Generating embeddings for multiple document chunks, summarizing multiple retrieved passages, processing multiple user queries simultaneously (e.g., in a dashboard or multi-user system), re-ranking retrieved documents.

By combining these strategies, you can build a highly optimized RAG system that delivers fast, accurate, and cost-effective responses, even under heavy load. The "Combined Conceptual Flow" illustrates how these techniques might be orchestrated within a single RAG query lifecycle.


### Resources

*   **`functools.lru_cache` Documentation:** [https://docs.python.org/3/library/functools.html#functools.lru_cache](https://docs.python.org/3/library/functools.html#functools.lru_cache)
*   **LangChain Caching Strategies:** [https://python.langchain.com/docs/modules/model_io/llms/llm_caching](https://python.langchain.com/docs/modules/model_io/llms/llm_caching)
*   **LlamaIndex Caching:** [https://docs.llamaindex.ai/en/stable/module_guides/supporting_modules/cache.html](https://docs.llamaindex.ai/en/stable/module_guides/supporting_modules/cache.html)
*   **OpenAI API Pricing:** [https://openai.com/pricing](https://openai.com/pricing) (Understand token costs for different models)
*   **Anthropic Claude Pricing:** [https://www.anthropic.com/api/pricing](https://www.anthropic.com/api/pricing)
*   **Google Gemini Pricing:** [https://cloud.google.com/vertex-ai/generative-ai/pricing](https://cloud.google.com/vertex-ai/generative-ai/pricing)
*   **Hugging Face Inference Endpoints (for batching/scaling):** [https://huggingface.co/docs/inference-endpoints/index](https://huggingface.co/docs/inference-endpoints/index)
*   **Blog Post: LLM Cost Optimization Strategies:** (Search for recent articles on platforms like Towards Data Science, Medium, or major cloud provider blogs for 2024-2026 insights on LLM cost management).
